In [4]:
import chess
import random
import json
from tqdm import tqdm

GAMES_TO_SIMULATE = 5000
MAX_MOVES_PER_GAME = 80
PARAPHRASES_PER_MOVE = 6
OUTPUT_FILE = "synthetic_chess_dataset.jsonl"

In [11]:
def generate_templates(board, move, san):
    piece = board.piece_at(move.from_square)
    target = chess.square_name(move.to_square)

    is_capture = board.is_capture(move)
    is_castle = board.is_castling(move)
    is_promotion = move.promotion is not None

    templates = []

    if is_castle:
        if chess.square_file(move.to_square) == 6:
            templates += [
                "Castle kingside",
                "Kingside castle",
                "Perform kingside castling",
                "King castles short",
                "Castle short",
                "Short castle",
            ]
        else:
            templates += [
                "Castle queenside",
                "Queenside castle",
                "Perform queenside castling",
                "King castles long",
                "Castle long",
                "Long castle",
            ]
        return templates

    if is_promotion:
        promo_piece = chess.piece_name(move.promotion)
        templates += [
            f"Promote pawn to {promo_piece} on {target}",
            f"Advance pawn to {target} and promote to {promo_piece}",
            f"Move pawn to {target} and make it a {promo_piece}",
        ]
        return templates

    p_name = chess.piece_name(piece.piece_type)

    if is_capture:
        templates += [
            f"{p_name.capitalize()} captures on {target}",
            f"Capture the piece on {target} with the {p_name}",
            f"Take on {target} using the {p_name}",
            f"Use the {p_name} to capture on {target}",
        ]
    elif piece.piece_type == chess.PAWN:
        templates += [
            f"Advance pawn to {target}",
            f"Push the pawn to {target}",
            f"Move pawn to {target}",
        ]
    else:
        templates += [
            f"Move the {p_name} to {target}",
            f"Play {p_name} to {target}",
            f"Develop the {p_name} to {target}",
            f"{p_name.capitalize()} goes to {target}",
        ]

    templates.append(f"Play {san}")

    return templates

def simulate_game():
    board = chess.Board()
    examples = []

    for _ in range(MAX_MOVES_PER_GAME):
        if board.is_game_over():
            break

        fen = board.fen()

        legal_moves = list(board.legal_moves)

        move = random.choice(legal_moves)
        san = board.san(move)

        templates = generate_templates(board, move, san)

        chosen_templates = random.sample(
            templates,
            min(len(templates), PARAPHRASES_PER_MOVE)
        )

        for instruction in chosen_templates:
            examples.append(
                {
                    "fen": fen,
                    "instruction": instruction,
                    "san": san
                }
            )

        board.push(move)

    return examples

def generate_dataset():
    with open(OUTPUT_FILE, "w") as f:
        for _ in tqdm(range(GAMES_TO_SIMULATE)):
            game_examples = simulate_game()
            
            for ex in game_examples:
                f.write(json.dumps(ex) + "\n")

In [12]:
if __name__ == "__main__":
    generate_dataset()
    print("Dataset generation complete.")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [02:11<00:00, 38.12it/s]

Dataset generation complete.
